# NeqSim Studio — Quick Start

**NeqSim Studio** is a Python-first, newcomer-friendly way to build NeqSim process models.
It wraps the trusted NeqSim Java engine with a small Python layer so you can go from an idea
to a running flowsheet in one or two lines.

This notebook demonstrates all five building methods:

1. **Natural language** — describe the process in plain English
2. **Template recipes** — standard unit trains with a few knobs
3. **Guided wizard** — answer a few questions
4. **Edit by chat / automation** — tweak a built model conversationally
5. **Recipe gallery** — copy-paste ready cookbook examples

Every builder produces the JSON understood by NeqSim's `JsonProcessBuilder` and runs through
`ProcessSystem.fromJsonAndRun`, so all the engineering stays in the trusted Java core.

## Setup

Use the standard NeqSim dev setup so workspace Java classes from `target/classes` are loaded,
then hand the namespace to `connect` so the JVM is reused (not restarted).

In [1]:
import os
import sys
from pathlib import Path


def find_neqsim_project_root():
    env_root = os.environ.get("NEQSIM_PROJECT_ROOT")
    candidates = []
    if env_root:
        candidates.append(Path(env_root).resolve())
    cwd = Path.cwd().resolve()
    candidates.extend([cwd] + list(cwd.parents))
    for candidate in candidates:
        if (candidate / "pom.xml").exists() and (candidate / "devtools" / "neqsim_dev_setup.py").exists():
            return candidate
    raise RuntimeError("Could not find NeqSim project root. Set NEQSIM_PROJECT_ROOT.")


PROJECT_ROOT = find_neqsim_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "devtools"))

from neqsim_dev_setup import neqsim_init, neqsim_classes

ns = neqsim_init(project_root=PROJECT_ROOT, recompile=False, verbose=True)
ns = neqsim_classes(ns)

from neqsim_studio import connect

studio = connect(ns=ns)
print("NeqSim Studio ready")

NeqSim project root: C:\Users\ESOL\Documents\GitHub\neqsim
Classpath:
  1. C:\Users\ESOL\Documents\GitHub\neqsim\target\classes
  2. C:\Users\ESOL\Documents\GitHub\neqsim\src\main\resources
  3. C:\Users\ESOL\.m2\repository\com\h2database\h2\2.4.240\h2-2.4.240.jar
  4. C:\Users\ESOL\.m2\repository\org\apache\logging\log4j\log4j-api\2.26.0\log4j-api-2.26.0.jar
  5. C:\Users\ESOL\.m2\repository\org\apache\logging\log4j\log4j-core\2.26.0\log4j-core-2.26.0.jar
  6. C:\Users\ESOL\.m2\repository\com\thoughtworks\xstream\xstream\1.4.21\xstream-1.4.21.jar
  7. C:\Users\ESOL\.m2\repository\io\github\x-stream\mxparser\1.2.2\mxparser-1.2.2.jar
  8. C:\Users\ESOL\.m2\repository\xmlpull\xmlpull\1.1.3.1\xmlpull-1.1.3.1.jar
  9. C:\Users\ESOL\.m2\repository\org\apache\commons\commons-lang3\3.20.0\commons-lang3-3.20.0.jar
  10. C:\Users\ESOL\.m2\repository\org\apache\commons\commons-math3\3.6.1\commons-math3-3.6.1.jar
  11. C:\Users\ESOL\.m2\repository\org\ejml\ejml-all\0.45.1\ejml-all-0.45.1.jar
  12


JVM started: C:\Users\ESOL\graalvm\graalvm-jdk-25.0.1+8.1\bin\server\jvm.dll
Ready — call neqsim_classes(ns) to import classes


All NeqSim classes imported OK
NeqSim Studio ready


## 1. Natural language

Describe the process in plain English. This works **offline** with a rule-based parser
(no LLM required). Feed conditions and fluid type are auto-detected.

In [2]:
result = studio.from_text(
    "Take natural gas at 25 C and 40 bara, cool it to 20 C, then compress to 150 bara"
)
result.summary()
print("units:", result.units())

NeqSim flowsheet: json-process  (source: text:rules)
------------------------------------------------------------
Equipment (3):
  - feed                         Stream
  - Cooler 1                     Cooler
  - Compressor 1                 Compressor
------------------------------------------------------------
Key results:
  total_compressor_power_MW              0.0506
  total_cooling_duty_MW                  0.0033
units: ['feed', 'Cooler 1', 'Compressor 1']


## 2. Template recipes

Parameterised standard trains. Pass only the knobs you care about; a preset feed fluid is
created automatically when you don't supply one.

In [3]:
comp = studio.gas_compression(
    suction_pressure_bara=20,
    discharge_pressure_bara=120,
    stages=2,
    interstage_temperature_c=30,
    polytropic_efficiency=0.78,
)
comp.summary()
print("key results:", comp.key_results())
print("available templates:", list(studio.list_templates().keys()))

NeqSim flowsheet: json-process  (source: template:gas_compression)
------------------------------------------------------------
Equipment (4):
  - feed                         Stream
  - Stage 1 Compressor           Compressor
  - Stage 1 Intercooler          Cooler
  - Stage 2 Compressor           Compressor
------------------------------------------------------------
Key results:
  total_compressor_power_MW             11.6405
  total_cooling_duty_MW                  8.6352
key results: {'total_compressor_power_MW': 11.640538204422253, 'total_cooling_duty_MW': 8.63519325794297}
available templates: ['gas_compression', 'three_stage_separation', 'dehydration', 'co2_capture']


In [4]:
sep = studio.three_stage_separation(hp_pressure_bara=70, mp_pressure_bara=20, lp_pressure_bara=3)
print("separation units:", sep.units())

separation units: ['feed', 'HP Separator', 'HP to MP Valve', 'MP Separator', 'MP to LP Valve', 'LP Separator']


## 3. Guided wizard

Answer a few questions and Studio routes them to the right recipe. Here we pass the answers
programmatically; set `interactive=True` to be prompted instead.

In [5]:
wiz = studio.wizard(answers={
    "objective": "compress",
    "fluid": "lean_gas",
    "feed_temperature_c": "35",
    "feed_pressure_bara": "30",
    "target_pressure_bara": "100",
})
print("wizard units:", wiz.units())
print("wizard key results:", wiz.key_results())

wizard units:

 ['feed', 'Stage 1 Compressor', 'Stage 1 Intercooler', 'Stage 2 Compressor']
wizard key results: {'total_compressor_power_MW': 7.569373994304553, 'total_cooling_duty_MW': 5.434854508570915}


## 4. Edit by chat / automation

Once a flowsheet is built, tweak it conversationally. The editor wraps the NeqSim
`ProcessAutomation` facade (string-addressable variables, self-healing fuzzy matching).

In [6]:
editor = comp.editor()
print("units:", editor.units())

out = editor.apply_command("set Stage 1 Compressor outlet pressure to 55 bara")
print("applied:", out.get("ok"), "->", out.get("address"))

print("new power:", comp.key_results())

units: ['feed', 'Stage 1 Compressor', 'Stage 1 Intercooler', 'Stage 2 Compressor']
applied: True -> Stage 1 Compressor.outletPressure
new power: {'total_compressor_power_MW': 11.672531912264713, 'total_cooling_duty_MW': 9.835693268931948}


## 5. Recipe gallery

A cookbook of self-contained, ready-to-run recipes (each carries its own fluid and JSON).

In [7]:
print("recipes:", list(studio.recipes().keys()))
gallery_result = studio.build_recipe("two_stage_export_compression")
gallery_result.summary()

recipes: ['two_stage_export_compression', 'three_stage_separation', 'water_knockout', 'co2_conditioning']
NeqSim flowsheet: json-process  (source: json)
------------------------------------------------------------
Equipment (4):
  - feed                         Stream
  - Stage 1 Compressor           Compressor
  - Stage 1 Intercooler          Cooler
  - Stage 2 Compressor           Compressor
------------------------------------------------------------
Key results:
  total_compressor_power_MW             11.6405
  total_cooling_duty_MW                  8.6352


'NeqSim flowsheet: json-process  (source: json)\n------------------------------------------------------------\nEquipment (4):\n  - feed                         Stream\n  - Stage 1 Compressor           Compressor\n  - Stage 1 Intercooler          Cooler\n  - Stage 2 Compressor           Compressor\n------------------------------------------------------------\nKey results:\n  total_compressor_power_MW             11.6405\n  total_cooling_duty_MW                  8.6352'

## Visualise the flowsheet

`result.show()` draws a simple block diagram from the topology.

In [8]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9, 3))
comp.show(ax=ax)
ax.set_title("Two-stage gas compression")
plt.tight_layout()
plt.show()

C:\Users\ESOL\AppData\Local\Temp\ipykernel_2728\4242077.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Where to go next

- Package README: `devtools/neqsim_studio/README.md`
- Reference page: `docs/process/neqsim-studio.md`
- The pure-Python building blocks (`FlowsheetSpec`, `parse_edit_command`, `text_to_json`,
  the `*_spec` template functions, the gallery, and the wizard router) are unit-tested
  without starting Java: `python devtools/test_neqsim_studio.py`